# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking
**Document:** MCC *Laws of Cricket* (2017 Code, 4th Edition, 2026)

This notebook builds two retrieval pipelines over the **same chunked corpus** and compares them:

| | Pipeline 1 - Baseline | Pipeline 2 - Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder (`all-MiniLM-L6-v2`) retrieves **Top 10** by cosine similarity | *same Top 10* |
| Stage 2 | - | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) scores all 10 (query, chunk) pairs and re-sorts |
| To LLM | Top 3 bi-encoder chunks → **Answer 1** | Top 3 re-ranked chunks → **Answer 2** |

Both pipelines use the **identical prompt** and the same LLM (Gemini, temperature 0).

**Contents**
1. Setup
2. Load and clean the PDF
3. Chunking
4. Stage 1 - Bi-encoder index (vector store)
5. Stage 2 - Cross-encoder re-ranker
6. LLM and shared prompt
7. The two pipelines end-to-end
8. Sanity check
9. Failure mode analysis
10. Rank-shift summary
11. Latency profiling
12. Scaling experiment

## 1. Setup
Install the dependencies once (uncomment the line below), then set your Gemini API key in a file called `.env` next to this notebook:
```
GEMINI_API_KEY=your-key-here
```
The key is read with `python-dotenv`, so it never appears in the notebook. `.env` is listed in `.gitignore` so it is not pushed to GitHub.

In [1]:
# !pip install pypdf sentence-transformers faiss-cpu google-genai python-dotenv pandas

In [2]:
import os
import re
import string
import time

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pypdf import PdfReader
from sentence_transformers import CrossEncoder, SentenceTransformer

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

# ---- Configuration -------------------------------------------------------
PDF_PATH = "Laws-of-Cricket-2017-Code-4th-Edition-2026_COMPLETE.pdf"
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # Stage 1 (bi-encoder)
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # Stage 2 (cross-encoder)
TOP_K = 10  # candidates retrieved by the bi-encoder
TOP_N = 3   # chunks given to the LLM

load_dotenv()  # loads GEMINI_API_KEY (and optionally GEMINI_MODEL) from .env
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
print("Gemini key found:", bool(GEMINI_API_KEY), "| model:", GEMINI_MODEL)

C:\Users\sures\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1016 22:05:10.326000 14852 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Gemini key found: True | model: gemini-3.5-flash


## 2. Load and clean the PDF
The PDF has 90 pages, and each PDF page holds **two printed pages side by side**.
- PDF pages 5-84 hold the actual rules (Preamble, Laws 1-42, Appendices A-D). The cover, preface and contents pages at the front, and the index at the back, are skipped because they would only add noisy, duplicate text.
- The first line of each page is the printed page-number pair (e.g. `24 25`), so it is removed.
- Text extraction puts spaces inside rule numbers (`6 .1` instead of `6.1`), and curly quotes/dashes appear. Both are normalised so section numbers can be detected reliably.

In [3]:
FIRST_PAGE, LAST_PAGE = 4, 83  # 0-based PDF page indices to keep

CHAR_MAP = {"\xa0": " ", "\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-"}


def clean_line(line):
    """Normalise one extracted line of text."""
    for bad, good in CHAR_MAP.items():
        line = line.replace(bad, good)
    line = re.sub(r"(?<=\d)\s*\.\s*(?=\d)", ".", line)    # '6 .1 .2' -> '6.1.2'
    line = re.sub(r"(?<=\b[A-E])\.\s+(?=\d)", ".", line)  # 'A.2. 8'  -> 'A.2.8'
    line = re.sub(r"(\w)' s\b", r"\1's", line)             # "Striker' s" -> "Striker's"
    return re.sub(r"[ \t]+", " ", line).strip()


def load_lines(path):
    """Return a list of (clean_line, pdf_page_number) for the rule pages."""
    reader = PdfReader(path)
    lines = []
    for p in range(FIRST_PAGE, LAST_PAGE + 1):
        page_lines = (reader.pages[p].extract_text() or "").split("\n")
        if page_lines and re.fullmatch(r"\s*\d+\s+\d+\s*", page_lines[0]):
            page_lines = page_lines[1:]  # drop the printed page-number header
        lines += [(c, p + 1) for c in map(clean_line, page_lines) if c]
    return lines


lines = load_lines(PDF_PATH)
print(f"{len(lines)} lines of text from PDF pages {FIRST_PAGE + 1}-{LAST_PAGE + 1}")
for l, p in lines[40:48]:
    print(f"p{p}: {l}")

4360 lines of text from PDF pages 5-84
p5: the umpires before the toss.
p5: 1.2.2 No player may be replaced after the nomination without the
p5: consent of the opposing captain. If such consent is granted,
p5: the captain of the team in which the player is being replaced
p5: shall inform the umpires and scorers.
p5: 1.2.3 Any replacement player shall be considered the same player
p5: as the nominated player they replaced for the purposes of
p5: these Laws.


## 3. Chunking
The Laws have a clear hierarchy: **Law** (e.g. `LAW 36 LEG BEFORE WICKET`) → **section** (e.g. `36.1 Out LBW`) → clauses (`36.1.1`, `36.1.2` ...).
Chunking strategy:
- **One chunk = one numbered section**, so a rule and its sub-clauses stay together.
- Sections longer than **180 words** are split into windows of 180 words with a **30-word overlap**. The bi-encoder truncates input at 256 word-pieces, so this keeps every chunk fully visible to it.
- Each chunk is prefixed with its Law and section title (e.g. `Law 36 Leg Before Wicket - 36.1 Out LBW: ...`) so it keeps its context after splitting.
- Section metadata (`section`, `page`) is stored so rank changes can be reported as e.g. *"Law 36.1 moved from rank 5 to rank 1"*.

In [4]:
LAW_RE = re.compile(r"^(LAW \d+|APPENDIX [A-E]|THE PREAMBLE)\b\s*(.*)$")
SECTION_RE = re.compile(r"^((?:\d+|[A-E])\.\d+)\s+([A-Z][^.]{0,80})$")  # e.g. '36.1 Out LBW'
UNIT_RE = re.compile(r"^\d+\.\d+\s+(m|ft|in|cm|mm|g|oz)\b")              # not a heading: '20.12 m in length'


def build_sections(lines):
    """Group lines into sections, remembering which Law each belongs to."""
    sections, law, current = [], "Preamble", None
    for line, page in lines:
        m = LAW_RE.match(line)
        if m:
            law = string.capwords(f"{m.group(1)} {m.group(2)}".strip())
            current = None
            continue
        m = SECTION_RE.match(line)
        is_heading = bool(m) and not UNIT_RE.match(line)
        if is_heading or current is None:
            sec, title = (m.group(1), m.group(2).strip()) if is_heading else ("", "")
            current = {"law": law, "section": sec, "title": title, "page": page, "body": []}
            sections.append(current)
            if is_heading:
                continue
        current["body"].append(line)
    return sections


def make_chunks(sections, max_words=180, overlap=30):
    """Turn sections into chunks of at most max_words words."""
    chunks = []
    for s in sections:
        header = f"{s['law']} - {s['section']} {s['title']}".strip(" -")
        words = " ".join(s["body"]).split()
        if not words:
            continue
        step = max_words - overlap
        starts = [0] if len(words) <= max_words else range(0, len(words) - overlap, step)
        for part, start in enumerate(starts, 1):
            chunks.append({
                "chunk_id": len(chunks),
                "law": s["law"],
                "section": s["section"] or s["law"],
                "title": s["title"],
                "page": s["page"],
                "part": part,
                "text": f"{header}: " + " ".join(words[start:start + max_words]),
            })
    return chunks


sections = build_sections(lines)
chunks_df = pd.DataFrame(make_chunks(sections))
word_counts = chunks_df["text"].str.split().str.len()
print(f"{len(sections)} sections -> {len(chunks_df)} chunks")
print(f"words per chunk: min {word_counts.min()}, median {int(word_counts.median())}, max {word_counts.max()}")
chunks_df[["chunk_id", "section", "title", "page", "part", "text"]].iloc[270:276]

331 sections -> 392 chunks
words per chunk: min 12, median 91, max 198


,chunk_id,section,title,page,part,text
270,270,35.2,Not out Hit wicket,55,1,Law 35 Hit Wicket - 35.2 Not out Hit wicket: The striker is not out under this Law should their wicket be ...
271,271,35.2,Not out Hit wicket,55,2,Law 35 Hit Wicket - 35.2 Not out Hit wicket: throw in at any time. the bowler after entering the delivery ...
272,272,35.3,Runs scored,55,1,Law 35 Hit Wicket - 35.3 Runs scored: No runs shall be scored when the striker is dismissed Hit wicket oth...
273,273,36.1,Out LBW,55,1,Law 36 Leg Before Wicket - 36.1 Out LBW: The striker is out LBW if all the circumstances set out in 36.1.1...
274,274,36.2,Interception of the ball,56,1,"Law 36 Leg Before Wicket - 36.2 Interception of the ball: 36.2.1 In assessing points in 36.1.3, 36.1.4 and..."
275,275,36.3,Off side of wicket,56,1,Law 36 Leg Before Wicket - 36.3 Off side of wicket: The off side of the striker's wicket shall be determin...


## 4. Stage 1 - Bi-encoder index (vector store)
The bi-encoder embeds every chunk **independently** into a 384-dimensional vector. The vectors are L2-normalised and stored in a FAISS `IndexFlatIP` (exact inner-product search). With normalised vectors, inner product = **cosine similarity**.

Chunk embeddings are computed **once, offline**. At query time only the query is embedded and compared against the stored vectors.

In [5]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(
    chunks_df["text"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False
).astype("float32")
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks (dim={chunk_embeddings.shape[1]}) in {time.perf_counter() - t0:.2f}s (one-off cost)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4262.97it/s]

Indexed 392 chunks (dim=384) in 5.03s (one-off cost)


In [6]:
def bi_encoder_search(query, k=TOP_K):
    """Stage 1: embed the query and return the top-k chunks by cosine similarity."""
    q_emb = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    hits = chunks_df.iloc[ids[0]].copy()
    hits["bi_score"] = scores[0]
    hits["bi_rank"] = np.arange(1, k + 1)
    return hits.reset_index(drop=True)

## 5. Stage 2 - Cross-encoder re-ranker
The cross-encoder reads the query **and** a chunk together in one input (`[CLS] query [SEP] chunk [SEP]`), so every query token can attend to every chunk token. It outputs one relevance score per pair. It is applied only to the 10 bi-encoder candidates.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)


def rerank(query, candidates):
    """Stage 2: score each (query, chunk) pair with the cross-encoder and re-sort."""
    pairs = [(query, text) for text in candidates["text"]]
    reranked = candidates.copy()
    reranked["ce_score"] = cross_encoder.predict(pairs)
    reranked = reranked.sort_values("ce_score", ascending=False).reset_index(drop=True)
    reranked["ce_rank"] = np.arange(1, len(reranked) + 1)
    return reranked

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3588.82it/s]

## 6. LLM and shared prompt
Both pipelines use exactly the same prompt template; only the 3 context passages differ.
The prompt tells the model to answer **only from the context** and to say so when the answer is missing. This matters because Gemini already knows cricket: without this instruction it could answer correctly from memory and hide a retrieval failure.
Temperature is 0 so answers are repeatable.

In [8]:
from google import genai
from google.genai import types

llm_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

PROMPT_TEMPLATE = """You answer questions about the MCC Laws of Cricket.
Use ONLY the context passages below. Do not use any outside knowledge of cricket.
If the passages do not contain the answer, reply exactly: "Not found in the provided context."
Cite the Law/section numbers (e.g. Law 36.1) that support your answer. Keep the answer under 120 words.

Context:
{context}

Question: {question}
Answer:"""


def build_prompt(question, top_chunks):
    """Fill the shared template with the top-N chunks."""
    context = "\n\n".join(
        f"[Passage {i}] {row.text}" for i, row in enumerate(top_chunks.itertuples(), 1)
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)


def generate_answer(prompt, retries=3):
    """Call Gemini (temperature 0). Returns (answer_text, seconds)."""
    if llm_client is None:
        return "[LLM skipped - GEMINI_API_KEY not set]", 0.0
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            response = llm_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0),
            )
            return response.text.strip(), time.perf_counter() - t0
        except Exception as e:  # rate limit (429) or model overloaded (503) -> wait and retry
            if any(code in str(e) for code in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")) and attempt < retries - 1:
                time.sleep(30)
                continue
            raise

In [9]:
# Optional: list the Gemini "flash" models your key can use (set GEMINI_MODEL in .env to change).
if llm_client:
    print([m.name for m in llm_client.models.list() if "flash" in m.name][:15])

['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview']


## 7. The two pipelines end-to-end
`run_both()` runs the bi-encoder search **once** and gives the same 10 candidates to both pipelines, so the only difference between Answer 1 and Answer 2 is the re-ranking step. Each stage is timed separately with `time.perf_counter()`.

In [10]:
def run_both(query, call_llm=True):
    """Run Pipeline 1 (baseline) and Pipeline 2 (re-ranking) for one query."""
    t0 = time.perf_counter()
    top10 = bi_encoder_search(query)                 # Stage 1 (shared)
    t_search = time.perf_counter() - t0

    t0 = time.perf_counter()
    reranked = rerank(query, top10)                  # Stage 2 (Pipeline 2 only)
    t_rerank = time.perf_counter() - t0

    top3_baseline = top10.head(TOP_N)
    top3_reranked = reranked.head(TOP_N)

    result = {"query": query, "top10": top10, "reranked": reranked,
              "t_search": t_search, "t_rerank": t_rerank,
              "answer1": None, "answer2": None, "t_llm1": None, "t_llm2": None}
    if call_llm:
        result["answer1"], result["t_llm1"] = generate_answer(build_prompt(query, top3_baseline))
        result["answer2"], result["t_llm2"] = generate_answer(build_prompt(query, top3_reranked))
    return result


def gold_mask(df, gold):
    """Gold entries are a section ('41.9', any part) or a section/part ('24.2/2', one chunk)."""
    keys = df["section"] + "/" + df["part"].astype(str)
    return df["section"].isin(gold) | keys.isin(gold)


def rank_shift_table(result, gold=()):
    """One row per candidate: bi-encoder rank vs cross-encoder rank. Gold chunks are marked."""
    t = result["reranked"][["chunk_id", "section", "part", "title", "bi_rank", "ce_rank", "bi_score", "ce_score"]].copy()
    t = t.sort_values("bi_rank").reset_index(drop=True)
    t["shift"] = [f"#{b} -> #{c}" for b, c in zip(t["bi_rank"], t["ce_rank"])]
    t["gold"] = np.where(gold_mask(t, gold), "<== GOLD", "")
    t["in_top3"] = [("P1 " if b <= TOP_N else "   ") + ("P2" if c <= TOP_N else "")
                    for b, c in zip(t["bi_rank"], t["ce_rank"])]
    return t[["bi_rank", "ce_rank", "shift", "section", "part", "title", "chunk_id",
              "bi_score", "ce_score", "in_top3", "gold"]].round(3)


def show_answers(result):
    display(Markdown(f"**Answer 1 (baseline top-3):**\n\n> {result['answer1']}"))
    display(Markdown(f"**Answer 2 (re-ranked top-3):**\n\n> {result['answer2']}"))

## 8. Sanity check
A simple query where both pipelines should agree. The answer is in Law 1.1.

In [11]:
sanity = run_both("How many players are there in each side?")
display(rank_shift_table(sanity, gold=["1.1"]))
show_answers(sanity)

,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,1,#1 -> #1,1.1,1,Number of players,2,0.585,5.817,P1 P2,<== GOLD
1,2,3,#2 -> #3,1.2,1,Nomination and replacement,386,0.436,-4.822,P1 P2,
2,3,2,#3 -> #2,A.7,1,Fielders,362,0.420,-2.005,P1 P2,
3,4,10,#4 -> #10,A.3,1,The playing area,355,0.400,-10.850,,
4,5,4,#5 -> #4,A.6,1,Batters,360,0.370,-5.529,,
5,6,9,#6 -> #9,8.1,1,"Description, width and position",53,0.337,-10.000,,
6,7,8,#7 -> #8,9.1,2,Rolling,59,0.322,-9.734,,
7,8,7,#8 -> #7,41.15,3,Striker in protected area,329,0.322,-9.367,,
8,9,6,#9 -> #6,16.1,1,A Win - two-innings match,108,0.322,-8.782,,
9,10,5,#10 -> #5,24.4,1,Player returning without permission,203,0.318,-8.388,,


**Answer 1 (baseline top-3):**

> According to Law 1.1, a match is played between two sides, each consisting of eleven players, one of whom shall be captain. However, by agreement, a match may be played between sides of fewer than, or more than, eleven players, but no more than eleven players may field at any time.

**Answer 2 (re-ranked top-3):**

> According to Law 1.1, a match is played between two sides, each consisting of eleven players, one of whom shall be captain. However, by agreement, a match may be played between sides of fewer than, or more than, eleven players.

## 9. Failure mode analysis
Each case lists the **gold section(s)** (`41.9` = every chunk of that section, `24.2/2` = only part 2 of it), meaning the part of the Laws that actually answers the question (checked by reading the PDF). The table marks where the gold chunk(s) sit in the bi-encoder ranking vs. the cross-encoder ranking.

Column `in_top3`: `P1` = chunk is in the baseline top-3, `P2` = chunk is in the re-ranked top-3.

In [12]:
FAILURE_CASES = [
    {"category": "Keyword trap / distractor",
     "query": "What is the penalty for the fielding side wasting time?",
     "gold": ["41.9"],
     "note": "Gold is Law 41.9 (Time wasting by the fielding side). Distractor: Law 41.10 (Batter wasting time) shares the words 'wasting time' and 'penalty'."},
    {"category": "Synonym gap / distractor",
     "query": "Can the fielding side use saliva or sweat to shine the ball?",
     "gold": ["41.3"],
     "note": "Gold is Law 41.3 (The match ball - changing its condition), which talks about polishing and 'natural substances'. Distractor: Law 9.4 (Watering the pitch)."},
    {"category": "Exception / corner case",
     "query": "Is a batter out LBW if the ball pitched outside leg stump?",
     "gold": ["36.1"],
     "note": "Gold is Law 36.1 (Out LBW): the ball must land in line or on the off side. Distractors are general 'out' rules for other dismissals."},
    {"category": "Negation / constraint",
     "query": "Can a fielder who has just come back onto the field bowl straight away?",
     "gold": ["24.2/2"],
     "note": "Gold is the Law 24.2 chunk saying the fielder 'shall not be permitted to bowl' until Penalty time is served."},
    {"category": "Buried chunk (ranks 4-10)",
     "query": "Are runs scored if the batters run after the ball hits the striker's body without any attempt to play it?",
     "gold": ["23.2"],
     "note": "Gold is Law 23.2 (Leg byes). The query never says 'leg bye'."},
    {"category": "Buried chunk (ranks 4-10)",
     "query": "Is a run counted if the batter fails to ground their bat beyond the popping crease when turning?",
     "gold": ["18.3"],
     "note": "Gold is Law 18.3 (Short runs). The query never says 'short run'."},
    {"category": "Limitation - re-ranker makes it worse",
     "query": "Is a bowler allowed to deliberately bowl a high full toss at the batter?",
     "gold": ["41.7"],
     "note": "Gold is Law 41.7 (Dangerous and unfair non-landing deliveries)."},
    {"category": "Limitation - gold not in top 10",
     "query": "Is a batter allowed to wilfully damage the ball?",
     "gold": ["41.3/1"],
     "note": "Gold is Law 41.3.2, in the first chunk of 41.3 ('a batter is not allowed to wilfully damage the ball')."},
]

results = []
for case in FAILURE_CASES:
    results.append(run_both(case["query"]))
print(f"Ran {len(results)} failure queries.")

Ran 8 failure queries.


### Case 1: Keyword trap - fielding side vs. batter wasting time

In [13]:
case, res = FAILURE_CASES[0], results[0]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Keyword trap / distractor | What is the penalty for the fielding side wasting time?
Gold is Law 41.9 (Time wasting by the fielding side). Distractor: Law 41.10 (Batter wasting time) shares the words 'wasting time' and 'penalty'.


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,9,#1 -> #9,37.5,1,Runs scored,280,0.682,-0.792,P1,
1,2,8,#2 -> #8,41.17,2,Penalty runs,332,0.660,0.303,P1,
2,3,5,#3 -> #5,41.15,2,Striker in protected area,328,0.655,0.836,P1,
3,4,4,#4 -> #4,41.10,1,Batter wasting time,317,0.654,2.812,,
4,5,2,#5 -> #2,41.10,2,Batter wasting time,318,0.650,6.323,P2,
5,6,6,#6 -> #6,18.5,2,Deliberate short runs,133,0.640,0.757,,
6,7,7,#7 -> #7,41.14,2,Batter damaging the pitch,325,0.636,0.367,,
7,8,3,#8 -> #3,41.9,1,Time wasting by the fielding side,315,0.635,5.878,P2,<== GOLD
8,9,10,#9 -> #10,41.17,1,Penalty runs,331,0.629,-3.954,,
9,10,1,#10 -> #1,41.9,2,Time wasting by the fielding side,316,0.626,6.545,P2,<== GOLD


**Answer 1 (baseline top-3):**

> Not found in the provided context.

**Answer 2 (re-ranked top-3):**

> For a first instance of time wasting, the bowler's end umpire gives the captain of the fielding side a first and final warning (Law 41.9.2). 

For any further time wasting in that innings by any fielder:
* If the waste of time is **not during an over**, 5 Penalty runs are awarded to the batting side (Law 41.9, Law 41.9.3).
* If the waste of time is **during an over**, the captain of the fielding side is directed to immediately suspend the bowler from bowling. The suspended bowler shall not bowl again in that innings (Law 41.9).

### Case 2: Synonym gap - saliva / sweat vs. 'natural substance'

In [14]:
case, res = FAILURE_CASES[1], results[1]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Synonym gap / distractor | Can the fielding side use saliva or sweat to shine the ball?
Gold is Law 41.3 (The match ball - changing its condition), which talks about polishing and 'natural substances'. Distractor: Law 9.4 (Watering the pitch).


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,9,#1 -> #9,9.4,1,Watering the pitch,63,0.519,-11.070,P1,
1,2,1,#2 -> #1,41.3,1,The match ball - changing its condition,298,0.495,-5.788,P1 P2,<== GOLD
2,3,4,#3 -> #4,28.3,2,Protective helmets belonging to the fielding side,232,0.478,-6.577,P1,
3,4,3,#4 -> #3,28.2,1,Fielding the ball,229,0.460,-6.501,P2,
4,5,2,#5 -> #2,28.3,1,Protective helmets belonging to the fielding side,231,0.456,-6.444,P2,
5,6,7,#6 -> #7,26.2,1,Practice on the outfield,216,0.446,-10.539,,
6,7,6,#7 -> #6,28.4,1,Limitation of on side fielders,233,0.440,-9.573,,
7,8,8,#8 -> #8,2.5,1,"Conduct of the match, implements and equipment",10,0.432,-11.038,,
8,9,5,#9 -> #5,26.3,1,Trial run-up,218,0.421,-9.030,,
9,10,10,#10 -> #10,9.8,1,Non-turf pitches,67,0.407,-11.279,,


**Answer 1 (baseline top-3):**

> Based on Law 41.3.2.1, a fielder may use sweat to polish the ball on their clothing, provided that no artificial substance is used and such polishing wastes no time. However, saliva cannot be used, as sweat is specified as "the only natural substance" permitted. 

Cites: Law 41.3.2.1

**Answer 2 (re-ranked top-3):**

> Based on Law 41.3.2.1, a fielder may use sweat to polish the ball on their clothing, provided that no artificial substance is used and the polishing wastes no time. However, saliva cannot be used, as sweat is explicitly stated to be the only natural substance allowed.

### Case 3: Exception - LBW and the ball pitching outside leg stump

In [15]:
case, res = FAILURE_CASES[2], results[2]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Exception / corner case | Is a batter out LBW if the ball pitched outside leg stump?
Gold is Law 36.1 (Out LBW): the ball must land in line or on the off side. Distractors are general 'out' rules for other dismissals.


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,4,#1 -> #4,22.9,1,Out from a Wide,191,0.639,1.129,P1,
1,2,9,#2 -> #9,21.7,1,"Ball bouncing more than once, rolling along the ground",172,0.627,-7.587,P1,
2,3,2,#3 -> #2,39.1,1,Out Stumped,289,0.616,1.843,P1 P2,
3,4,5,#4 -> #5,38.2,1,Batter not out Run out,283,0.614,0.245,,
4,5,1,#5 -> #1,36.1,1,Out LBW,273,0.613,4.814,P2,<== GOLD
5,6,3,#6 -> #3,39.3,1,Not out Stumped,291,0.610,1.587,P2,
6,7,7,#7 -> #7,32.1,1,Out Bowled,255,0.599,-3.628,,
7,8,6,#8 -> #6,38.4,1,Which batter is out,286,0.594,-2.840,,
8,9,10,#9 -> #10,38.3,2,Non-striker leaving their ground early,285,0.592,-8.996,,
9,10,8,#10 -> #8,38.5,1,Runs scored,287,0.591,-6.412,,


**Answer 1 (baseline top-3):**

> Not found in the provided context.

**Answer 2 (re-ranked top-3):**

> No. According to Law 36.1 and Law 36.1.2, for a striker to be out LBW, the ball (if it is not intercepted without landing) must first land "in line between wicket and wicket or on the off side of the striker's wicket." Because a ball pitching outside leg stump does not meet this requirement, the striker is not out LBW.

### Case 4: Negation - fielder returning to the field cannot bowl immediately

In [16]:
case, res = FAILURE_CASES[3], results[3]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Negation / constraint | Can a fielder who has just come back onto the field bowl straight away?
Gold is the Law 24.2 chunk saying the fielder 'shall not be permitted to bowl' until Penalty time is served.


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,3,#1 -> #3,24.2,1,Fielder absent or leaving the field of play,197,0.675,-0.411,P1 P2,
1,2,2,#2 -> #2,37.4,1,Returning the ball to a fielder,279,0.608,-0.052,P1 P2,
2,3,1,#3 -> #1,24.2,2,Fielder absent or leaving the field of play,198,0.592,2.076,P1 P2,<== GOLD
3,4,6,#4 -> #6,28.4,1,Limitation of on side fielders,233,0.570,-3.989,,
4,5,7,#5 -> #7,A.7,2,Fielders,363,0.558,-4.144,,
5,6,9,#6 -> #9,24.2,5,Fielder absent or leaving the field of play,201,0.553,-4.604,,
6,7,10,#7 -> #10,26.2,1,Practice on the outfield,216,0.547,-6.462,,
7,8,8,#8 -> #8,28.3,2,Protective helmets belonging to the fielding side,232,0.540,-4.565,,
8,9,4,#9 -> #4,24.3,1,Penalty time not incurred,202,0.538,-1.895,,
9,10,5,#10 -> #5,28.2,1,Fielding the ball,229,0.535,-2.815,,


**Answer 1 (baseline top-3):**

> No. A fielder who has returned to the field of play is not permitted to bowl until they have been back on the field for a period of time known as Penalty time, and all of their Penalty time has been served (Law 24.2.2.3 and Law 24.2.5).

**Answer 2 (re-ranked top-3):**

> No. A fielder who has been absent or left the field shall not be permitted to bowl until they have been back on the field of play for a period of time known as Penalty time (Law 24.2 and Law 24.2.2.3). Furthermore, Law 24.2.5 explicitly states that the player shall not bowl until all of their Penalty time has been served.

### Case 5: Buried chunk - leg byes

In [17]:
case, res = FAILURE_CASES[4], results[4]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Buried chunk (ranks 4-10) | Are runs scored if the batters run after the ball hits the striker's body without any attempt to play it?
Gold is Law 23.2 (Leg byes). The query never says 'leg bye'.


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,2,#1 -> #2,25.7,1,Restriction on the striker's runner,212,0.692,5.901,P1 P2,
1,2,8,#2 -> #8,18.8,1,Runs scored when a batter is dismissed,136,0.689,4.070,P1,
2,3,3,#3 -> #3,38.5,1,Runs scored,287,0.674,5.262,P1 P2,
3,4,6,#4 -> #6,21.15,1,Runs resulting from a No ball - how scored,180,0.672,4.455,,
4,5,1,#5 -> #1,23.2,1,Leg byes,193,0.659,6.389,P2,<== GOLD
5,6,9,#6 -> #9,18.9,1,Runs scored when the ball becomes dead other than at,137,0.648,3.506,,
6,7,4,#7 -> #4,37.5,1,Runs scored,280,0.647,5.261,,
7,8,5,#8 -> #5,25.8,1,Striker's right to play the ball,214,0.642,4.824,,
8,9,10,#9 -> #10,34.1,1,Out Hit the ball twice,263,0.641,1.105,,
9,10,7,#10 -> #7,18.10,1,Crediting of runs scored,138,0.639,4.072,,


**Answer 1 (baseline top-3):**

> Not found in the provided context.

**Answer 2 (re-ranked top-3):**

> No. According to Law 23.2.1, if a ball delivered by the bowler first strikes the person of the striker, runs shall be scored only if the umpire is satisfied that the striker has either attempted to play the ball with the bat or tried to avoid being hit by the ball.

### Case 6: Buried chunk - short runs

In [18]:
case, res = FAILURE_CASES[5], results[5]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Buried chunk (ranks 4-10) | Is a run counted if the batter fails to ground their bat beyond the popping crease when turning?
Gold is Law 18.3 (Short runs). The query never says 'short run'.


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,2,#1 -> #2,30.1,1,When out of their ground,243,0.690,5.554,P1 P2,
1,2,3,#2 -> #3,25.7,1,Restriction on the striker's runner,212,0.617,4.326,P1 P2,
2,3,5,#3 -> #5,38.4,1,Which batter is out,286,0.605,-1.905,P1,
3,4,1,#4 -> #1,18.3,1,Short runs,130,0.604,6.368,P2,<== GOLD
4,5,9,#5 -> #9,18.8,1,Runs scored when a batter is dismissed,136,0.604,-3.224,,
5,6,8,#6 -> #8,38.5,1,Runs scored,287,0.603,-2.756,,
6,7,6,#7 -> #6,30.2,2,Which is a batter's ground,245,0.594,-2.120,,
7,8,4,#8 -> #4,38.1,1,Out Run out,282,0.584,-1.347,,
8,9,7,#9 -> #7,25.6,2,Dismissal and conduct of a batter and their runner,210,0.579,-2.397,,
9,10,10,#10 -> #10,34.4,1,Runs permitted from ball lawfully struck more than once,266,0.570,-5.120,,


**Answer 1 (baseline top-3):**

> Not found in the provided context.

**Answer 2 (re-ranked top-3):**

> Not found in the provided context.

### Case 7: Limitation - re-ranker demotes the right chunk

In [19]:
case, res = FAILURE_CASES[6], results[6]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Limitation - re-ranker makes it worse | Is a bowler allowed to deliberately bowl a high full toss at the batter?
Gold is Law 41.7 (Dangerous and unfair non-landing deliveries).


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,9,#1 -> #9,41.6,1,Bowling of dangerous and unfair short deliveries,307,0.607,-3.488,P1,
1,2,3,#2 -> #3,41.13,2,Bowler running on protected area,323,0.605,1.750,P1 P2,
2,3,5,#3 -> #5,41.7,4,Bowling of dangerous and unfair non-landing deliveries,313,0.597,0.989,P1,<== GOLD
3,4,4,#4 -> #4,41.6,2,Bowling of dangerous and unfair short deliveries,308,0.596,1.325,,
4,5,2,#5 -> #2,41.8,1,Bowling of deliberate front-foot No ball,314,0.594,2.583,P2,
5,6,8,#6 -> #8,26.3,1,Trial run-up,218,0.590,-1.685,,
6,7,7,#7 -> #7,41.7,3,Bowling of dangerous and unfair non-landing deliveries,312,0.582,0.449,,<== GOLD
7,8,6,#8 -> #6,41.7,2,Bowling of dangerous and unfair non-landing deliveries,311,0.582,0.772,,<== GOLD
8,9,10,#9 -> #10,41.7,1,Bowling of dangerous and unfair non-landing deliveries,310,0.575,-4.249,,<== GOLD
9,10,1,#10 -> #1,41.3,4,The match ball - changing its condition,301,0.560,2.784,P2,


**Answer 1 (baseline top-3):**

> Not found in the provided context.

**Answer 2 (re-ranked top-3):**

> Not found in the provided context.

### Case 8: Limitation - the right chunk never reaches the top 10

In [20]:
case, res = FAILURE_CASES[7], results[7]
print(case["category"], "|", case["query"])
print(case["note"])
display(rank_shift_table(res, case["gold"]))
show_answers(res)

Limitation - gold not in top 10 | Is a batter allowed to wilfully damage the ball?
Gold is Law 41.3.2, in the first chunk of 41.3 ('a batter is not allowed to wilfully damage the ball').


,bi_rank,ce_rank,shift,section,part,title,chunk_id,bi_score,ce_score,in_top3,gold
0,1,2,#1 -> #2,41.14,3,Batter damaging the pitch,326,0.633,2.858,P1 P2,
1,2,6,#2 -> #6,5.5,1,Damage to the ball,38,0.632,-0.368,P1,
2,3,10,#3 -> #10,28.3,2,Protective helmets belonging to the fielding side,232,0.612,-5.219,P1,
3,4,1,#4 -> #1,41.14,1,Batter damaging the pitch,324,0.610,4.504,P2,
4,5,8,#5 -> #8,21.17,1,Out from a No ball,182,0.592,-2.234,,
5,6,9,#6 -> #9,28.3,1,Protective helmets belonging to the fielding side,231,0.587,-4.892,,
6,7,7,#7 -> #7,34.4,1,Runs permitted from ball lawfully struck more than once,266,0.578,-0.967,,
7,8,5,#8 -> #5,37.2,1,Not out Obstructing the field,277,0.576,0.187,,
8,9,4,#9 -> #4,34.1,1,Out Hit the ball twice,263,0.575,0.305,,
9,10,3,#10 -> #3,41.12,2,Fielder damaging the pitch,321,0.570,0.807,P2,


**Answer 1 (baseline top-3):**

> Not found in the provided context.

**Answer 2 (re-ranked top-3):**

> Not found in the provided context.

## 10. Rank-shift summary
For each query: the best rank of any gold chunk before and after re-ranking, and whether a gold chunk reached the top 3 (the chunks the LLM actually sees).

In [21]:
def best_rank(df, col, gold):
    ranks = df.loc[gold_mask(df, gold), col]
    return int(ranks.min()) if len(ranks) else None


rows = []
for case, res in zip(FAILURE_CASES, results):
    b = best_rank(res["reranked"], "bi_rank", case["gold"])
    c = best_rank(res["reranked"], "ce_rank", case["gold"])
    rows.append({
        "category": case["category"],
        "query": case["query"],
        "gold": ", ".join(case["gold"]),
        "bi_rank": b if b else "not in top 10",
        "ce_rank": c if c else "not in top 10",
        "shift": f"#{b} -> #{c}" if b else "-",
        "gold_in_top3_P1": bool(b and b <= TOP_N),
        "gold_in_top3_P2": bool(c and c <= TOP_N),
    })
summary_df = pd.DataFrame(rows)
display(summary_df)
print(f"Gold chunk in top-3: Pipeline 1 = {summary_df.gold_in_top3_P1.sum()}/{len(summary_df)}, "
      f"Pipeline 2 = {summary_df.gold_in_top3_P2.sum()}/{len(summary_df)}")

,category,query,gold,bi_rank,ce_rank,shift,gold_in_top3_P1,gold_in_top3_P2
0,Keyword trap / distractor,What is the penalty for the fielding side wasting time?,41.9,8,1,#8 -> #1,False,True
1,Synonym gap / distractor,Can the fielding side use saliva or sweat to shine the ball?,41.3,2,1,#2 -> #1,True,True
2,Exception / corner case,Is a batter out LBW if the ball pitched outside leg stump?,36.1,5,1,#5 -> #1,False,True
3,Negation / constraint,Can a fielder who has just come back onto the field bowl straight away?,24.2/2,3,1,#3 -> #1,True,True
4,Buried chunk (ranks 4-10),Are runs scored if the batters run after the ball hits the striker's body without any attempt to play it?,23.2,5,1,#5 -> #1,False,True
5,Buried chunk (ranks 4-10),Is a run counted if the batter fails to ground their bat beyond the popping crease when turning?,18.3,4,1,#4 -> #1,False,True
6,Limitation - re-ranker makes it worse,Is a bowler allowed to deliberately bowl a high full toss at the batter?,41.7,3,5,#3 -> #5,True,False
7,Limitation - gold not in top 10,Is a batter allowed to wilfully damage the ball?,41.3/1,not in top 10,not in top 10,-,False,False


Gold chunk in top-3: Pipeline 1 = 3/8, Pipeline 2 = 6/8


## 11. Latency profiling
Retrieval stages are fast and vary run to run, so each query is run `N_RUNS` times (after one warm-up run) and the mean is reported. The LLM is **not** re-called here, to save API quota; its time comes from the calls made in Section 9.

- **Pipeline 1** = bi-encoder search + LLM
- **Pipeline 2** = bi-encoder search + cross-encoder re-rank (10 pairs) + LLM

In [22]:
N_RUNS = 5
queries = [c["query"] for c in FAILURE_CASES]
run_both(queries[0], call_llm=False)  # warm-up (first call is slower)

timing_rows = []
for q in queries:
    for _ in range(N_RUNS):
        r = run_both(q, call_llm=False)
        timing_rows.append({"query": q, "search_ms": r["t_search"] * 1000, "rerank_ms": r["t_rerank"] * 1000})
timing_df = pd.DataFrame(timing_rows)

llm_times = [t for r in results for t in (r["t_llm1"], r["t_llm2"]) if t]
llm_ms = np.mean(llm_times) * 1000 if llm_times else float("nan")
search_ms, rerank_ms = timing_df["search_ms"].mean(), timing_df["rerank_ms"].mean()

latency_table = pd.DataFrame({
    "Stage": ["Bi-encoder search (query embed + top-10)", "Cross-encoder re-rank (10 pairs)",
              "LLM generation (Gemini)", "Retrieval only", "End-to-end (incl. LLM)"],
    "Pipeline 1 - Baseline (ms)": [search_ms, 0, llm_ms, search_ms, search_ms + llm_ms],
    "Pipeline 2 - Re-rank (ms)": [search_ms, rerank_ms, llm_ms, search_ms + rerank_ms, search_ms + rerank_ms + llm_ms],
}).round(1)
display(latency_table)
print(f"Re-ranking adds {rerank_ms:.1f} ms per query "
      f"({(search_ms + rerank_ms) / search_ms:.1f}x the retrieval time of the baseline).")

,Stage,Pipeline 1 - Baseline (ms),Pipeline 2 - Re-rank (ms)
0,Bi-encoder search (query embed + top-10),9.6,9.6
1,Cross-encoder re-rank (10 pairs),0.0,211.8
2,LLM generation (Gemini),5393.7,5393.7
3,Retrieval only,9.6,221.3
4,End-to-end (incl. LLM),5403.3,5615.1


Re-ranking adds 211.8 ms per query (23.1x the retrieval time of the baseline).


In [23]:
# Per-query breakdown (mean over N_RUNS)
timing_df.groupby("query")[["search_ms", "rerank_ms"]].mean().round(1)

,search_ms,rerank_ms
query,,
Are runs scored if the batters run after the ball hits the striker's body without any attempt to play it?,11.0,244.2
Can a fielder who has just come back onto the field bowl straight away?,9.4,244.0
Can the fielding side use saliva or sweat to shine the ball?,8.7,198.4
Is a batter allowed to wilfully damage the ball?,8.3,199.8
Is a batter out LBW if the ball pitched outside leg stump?,9.2,204.0
Is a bowler allowed to deliberately bowl a high full toss at the batter?,9.6,198.9
Is a run counted if the batter fails to ground their bat beyond the popping crease when turning?,11.0,217.5
What is the penalty for the fielding side wasting time?,9.4,187.4


## 12. Scaling experiment
What if we skipped the bi-encoder and ran the cross-encoder over **every** chunk? A cross-encoder has to run a full transformer pass for every (query, chunk) pair **at query time**, and nothing can be pre-computed. The bi-encoder's chunk vectors are computed once in advance, so a query only needs one encoding plus a fast vector search.

In [24]:
q = FAILURE_CASES[0]["query"]
all_pairs = [(q, t) for t in chunks_df["text"]]

t0 = time.perf_counter()
cross_encoder.predict(all_pairs, batch_size=64)
t_ce_all = time.perf_counter() - t0

t0 = time.perf_counter()
bi_encoder_search(q)
t_bi = time.perf_counter() - t0

per_pair_ms = t_ce_all / len(all_pairs) * 1000
scaling = pd.DataFrame({
    "Approach": ["Bi-encoder search over all chunks", "Cross-encoder over top 10 only",
                 f"Cross-encoder over ALL {len(chunks_df)} chunks"],
    "Time (ms)": [t_bi * 1000, rerank_ms, t_ce_all * 1000],
}).round(1)
display(scaling)

print(f"Cross-encoder cost: {per_pair_ms:.1f} ms per (query, chunk) pair on this machine")
for n in [10_000, 1_000_000]:
    secs = n * per_pair_ms / 1000
    print(f"Estimated cross-encoder time for {n:>9,} chunks: {secs:>8,.0f} s  (= {secs / 60:,.1f} min) per query")

,Approach,Time (ms)
0,Bi-encoder search over all chunks,7.6
1,Cross-encoder over top 10 only,211.8
2,Cross-encoder over ALL 392 chunks,5060.9


Cross-encoder cost: 12.9 ms per (query, chunk) pair on this machine
Estimated cross-encoder time for    10,000 chunks:      129 s  (= 2.2 min) per query
Estimated cross-encoder time for 1,000,000 chunks:   12,911 s  (= 215.2 min) per query
